In [1]:
import poolparty as pp
pp.init()

Party(outputs=[])

In [2]:
# Default behavior: mark_changes=False
pool = pp.from_iupac_motif('ACGTNNN', mode='sequential')
pool.generate_library(num_seqs=3)

,seq,pool[0].seq,pool[0].state,op[0]:from_iupac_motif.state,op[0]:from_iupac_motif.key.iupac_state
0,ACGTAAA,ACGTAAA,0,0,0
1,ACGTAAC,ACGTAAC,1,1,1
2,ACGTAAG,ACGTAAG,2,2,2


In [3]:
# Set mark_changes=True as party default
pp.set_default('mark_changes', True)
pool = pp.from_iupac_motif('ACGTNNN', mode='sequential')
pool.generate_library(num_seqs=3)

,seq,pool[1].seq,pool[1].state,op[1]:from_iupac_motif.state,op[1]:from_iupac_motif.key.iupac_state
0,ACGTaaa,ACGTaaa,0,0,0
1,ACGTaac,ACGTaac,1,1,1
2,ACGTaag,ACGTaag,2,2,2


In [4]:
# Explicit parameter overrides party default
pool = pp.from_iupac_motif('ACGTNNN', mode='sequential', mark_changes=False)
pool.generate_library(num_seqs=3)

,seq,pool[2].seq,pool[2].state,op[2]:from_iupac_motif.state,op[2]:from_iupac_motif.key.iupac_state
0,ACGTAAA,ACGTAAA,0,0,0
1,ACGTAAC,ACGTAAC,1,1,1
2,ACGTAAG,ACGTAAG,2,2,2


In [5]:
# Works with mutagenize() too
pp.init()
pp.set_default('mark_changes', True)
mutants = pp.mutagenize('ACGTACGT', num_mutations=2, mode='sequential')
mutants.generate_library(num_seqs=5)

,seq,pool[1].seq,op[0]:from_seq.key.seq,pool[0].seq,pool[1].state,pool[0].state,op[1]:mutagenize.state,op[0]:from_seq.state,op[1]:mutagenize.key.positions,op[1]:mutagenize.key.wt_chars,op[1]:mutagenize.key.mut_chars,op[0]:from_seq.key.seq
0,caGTACGT,caGTACGT,ACGTACGT,ACGTACGT,0,0,0,0,"(0, 1)","(A, C)","(c, a)",ACGTACGT
1,cgGTACGT,cgGTACGT,ACGTACGT,ACGTACGT,1,0,1,0,"(0, 1)","(A, C)","(c, g)",ACGTACGT
2,ctGTACGT,ctGTACGT,ACGTACGT,ACGTACGT,2,0,2,0,"(0, 1)","(A, C)","(c, t)",ACGTACGT
3,gaGTACGT,gaGTACGT,ACGTACGT,ACGTACGT,3,0,3,0,"(0, 1)","(A, C)","(g, a)",ACGTACGT
4,ggGTACGT,ggGTACGT,ACGTACGT,ACGTACGT,4,0,4,0,"(0, 1)","(A, C)","(g, g)",ACGTACGT


In [8]:
# Load defaults from a TOML file
import tempfile, os
with tempfile.NamedTemporaryFile(mode='w', suffix='.toml', delete=False) as f:
    f.write('mark_changes = true\n')
    toml_path = f.name

pp.init()
pp.load_defaults(toml_path)
pool = pp.from_iupac_motif('ACGTNNN', mode='sequential')
df = pool.generate_library(num_seqs=3)
os.unlink(toml_path)
df

,seq,pool[0].seq,pool[0].state,op[0]:from_iupac_motif.state,op[0]:from_iupac_motif.key.iupac_state
0,ACGTaaa,ACGTaaa,0,0,0
1,ACGTaac,ACGTaac,1,1,1
2,ACGTaag,ACGTaag,2,2,2


In [9]:
# Default party context demo - no explicit 'with pp.Party()' needed!
import poolparty as pp

# Create pools directly - works immediately after import
pool = pp.from_seqs(['AAA', 'TTT', 'GGG'], mode='sequential').named('seq')
df = pool.generate_library(num_seqs=3)
df

,seq,seq.seq,seq.state,seq.op.state,seq.op.key.seq_name,seq.op.key.seq_index
0,AAA,AAA,0,0,seq_0,0
1,TTT,TTT,1,1,seq_1,1
2,GGG,GGG,2,2,seq_2,2


In [10]:
# More complex operations also work without explicit context
mutants = pp.mutagenize('ACGT', num_mutations=1, mode='sequential').named('mutant')
df = mutants.generate_library(num_seqs=5)
df

,seq,mutant.seq,op[2]:from_seq.key.seq,pool[2].seq,mutant.state,pool[2].state,mutant.op.state,op[2]:from_seq.state,mutant.op.key.positions,mutant.op.key.wt_chars,mutant.op.key.mut_chars,op[2]:from_seq.key.seq
0,cCGT,cCGT,ACGT,ACGT,0,0,0,0,"(0,)","(A,)","(c,)",ACGT
1,gCGT,gCGT,ACGT,ACGT,1,0,1,0,"(0,)","(A,)","(g,)",ACGT
2,tCGT,tCGT,ACGT,ACGT,2,0,2,0,"(0,)","(A,)","(t,)",ACGT
3,AaGT,AaGT,ACGT,ACGT,3,0,3,0,"(1,)","(C,)","(a,)",ACGT
4,AgGT,AgGT,ACGT,ACGT,4,0,4,0,"(1,)","(C,)","(g,)",ACGT


In [11]:
# Join pools and use markers without context manager
left = pp.from_seqs(['AAA'])
right = pp.from_seqs(['TTT'])
oligo = pp.join([left, '---', right]).named('oligo')
df = oligo.generate_library(num_seqs=1)
df

,seq,oligo.seq,pool[5].seq,pool[4].seq,op[6]:from_seq.key.seq,pool[6].seq,oligo.state,pool[5].state,pool[4].state,pool[6].state,oligo.op.state,op[5]:from_seqs.state,op[4]:from_seqs.state,op[6]:from_seq.state,op[5]:from_seqs.key.seq_name,op[5]:from_seqs.key.seq_index,op[4]:from_seqs.key.seq_name,op[4]:from_seqs.key.seq_index,op[6]:from_seq.key.seq
0,AAA---TTT,AAA---TTT,TTT,AAA,---,---,0,0,0,0,0,0,0,0,seq_0,0,seq_0,0,---


In [12]:
# reset() clears state and optionally changes alphabet
pp.reset(alphabet='rna')
rna_pool = pp.from_seqs(['AAA', 'UUU'], mode='sequential').named('rna_seq')
df = rna_pool.generate_library(num_seqs=2)
df

,seq,rna_seq.seq,rna_seq.state,rna_seq.op.state,rna_seq.op.key.seq_name,rna_seq.op.key.seq_index
0,AAA,AAA,0,0,seq_0,0
1,UUU,UUU,1,1,seq_1,1


In [13]:
# Explicit Party context still works - temporarily replaces default
pp.init()  # back to DNA
with pp.Party(alphabet='protein') as party:
    protein_pool = pp.get_kmers(length=2, mode='sequential').named('dipeptide')
    df = protein_pool.generate_library(num_seqs=5)
    print(f"Inside explicit context: {pp.get_active_party().alphabet.chars[:5]}...")
print(f"After exit: {pp.get_active_party().alphabet.chars[:4]}")
df

Inside explicit context: ['A', 'C', 'D', 'E', 'F']...
After exit: ['A', 'C', 'G', 'T']


,seq,dipeptide.seq,dipeptide.state,dipeptide.op.state,dipeptide.op.key.kmer_index
0,AA,AA,0,0,0
1,AC,AC,1,1,1
2,AD,AD,2,2,2
3,AE,AE,3,3,3
4,AF,AF,4,4,4


In [14]:
# Nested contexts now work (they stack)
pp.init()
with pp.Party(alphabet='dna') as outer:
    print(f"Outer party active, alphabet: {outer.alphabet.chars}")
    with pp.Party(alphabet='rna') as inner:
        print(f"Inner party active, alphabet: {inner.alphabet.chars}")
    print(f"Back to outer, alphabet: {pp.get_active_party().alphabet.chars}")
print(f"Back to default, alphabet: {pp.get_active_party().alphabet.chars}")

Outer party active, alphabet: ['A', 'C', 'G', 'T']
Inner party active, alphabet: ['A', 'C', 'G', 'U']
Back to outer, alphabet: ['A', 'C', 'G', 'T']
Back to default, alphabet: ['A', 'C', 'G', 'T']
